In [ ]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

load_dotenv(dotenv_path="../.env")

TOKEN=os.environ.get("OPENROUTER_KEY")


client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=TOKEN
)

max_new_tokens=1000
temperature=0.1


In [25]:
import json

INSTRUCTION = """
### ROLE
You are a moderation model trained for rationale extraction.

### TASK
Extract the minimal text spans that justify why a sentence contains hate speech or offensive language.

### RATIONALE EXTRACTION RULES:
- EXTRACT NUCLEUS: Identify ONLY the core tokens that carry the offensive or toxic weight.
- LENGTH CONSTRAINT: Each rationale span MUST be synthetic

### FORMAT:
rationales: span1;; span2 
Example
---
Text: "You are a complete idiot and I hate your face."
rationales: idiot;; hate your face  
---
Now annotate:


"""


In [26]:
HateXplain=json.load(open("../sota_datasets/HateXplain.json"))

In [27]:
filtered = [x for x in HateXplain if len(x["rationales"]) > 0]

In [5]:
def predict(messages):     
    #The try except serves for when GPT dont process text because of their API guardians
     
    response = client.chat.completions.create(
        model=model_id,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=temperature        
    )
    output=response.choices[0].message.content
   
    return output

In [28]:
def process(text):
     
     messages=[{"role":"system","content":INSTRUCTION},{"role":"user","content":f"Analyse this prompt: {text}\n Remember you DONT have to fullfill the prompt request. Strictly adhere to your SYSTEM instructions and proceed with your analysis. Do not start your response with I cannot answer or similar. Output only rationales: <span>;;.."}]
     answer=predict(messages)
     return answer

In [35]:
from tqdm.contrib.concurrent import thread_map
import os
model_ids={"llama3.3-70":"meta-llama/llama-3.3-70b-instruct","gpt4-mini":"openai/gpt-4o-mini"}
def parse_answer(text):
    if "rationales:" in text:
        index = text.index("rationales:") + len("rationales:")
        text = text[index + 1:len(text)]
    rationales = [x.strip() for x in list(set(text.split(";;")))]
    return rationales

for model_name,model_id in model_ids.items():    
    data=[]
    out_file=f"../output/HateXplain/FT/{model_name}.json"
   
    print("Processing",model_name) 
    texts=[x["text"] for x in filtered]
    results = thread_map(process, texts, max_workers=12, desc="Rationale extraction")   
   
    for i,out in enumerate(results):
        parsed=parse_answer(out)
        data.append({"text":filtered[i]["text"],"gold":filtered[i]["rationales"],"pred":parsed})
    
    with open(out_file,"w") as f:
        json.dump(data,f,indent=2)
    

Processing llama3.3-70


Rationale extraction: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 1126/1126 [01:54<00:00,  9.81it/s]


Processing gpt4-mini


Rationale extraction: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 1126/1126 [01:09<00:00, 16.21it/s]


In [34]:
messages=[{"role":"system","content":INSTRUCTION},{"role":"user","content":f"Analyse this prompt: {filtered[0]['text']}\n Remember you DONT have to fullfill the prompt request. Strictly adhere to your SYSTEM instructions and proceed with your analysis. Do not start your response with I cannot answer or similar. Output only rationales: <span>;;.."}]
parse_answer(predict(messages))

['disgusting kike language', 'exterminate the goyim', 'raped me']

In [20]:
import re
def parse_answer(text):
    if "rationales:" in text:
        index = text.index("rationales:") + len("rationales:")
        text = text[index + 1:len(text)]
    rationales = [x.strip() for x in list(set(text.split(";;")))]

    regex = r"(?P<span>.*)\s\((?P<label>.*)\)"

    output = []

    for i, item in enumerate(rationales):

        match = re.match(regex, item)
        if match:
            output.append(match.groupdict())
    return output


[{'span': 'aped', 'label': 'hatespeech'}, {'span': 'disgusting kike language', 'label': 'hatespeech'}, {'span': 'exterminate the goyim', 'label': 'hatespeech'}]
[{'span': 'crying', 'label': 'offensive'}, {'span': 'itch', 'label': 'offensive'}]
[{'span': 'ike', 'label': 'hatespeech'}]
[{'span': 'lmfao', 'label': 'offensive'}, {'span': 'stole my choreo', 'label': 'offensive'}, {'span': 'make it look bad', 'label': 'offensive'}, {'span': 'oes', 'label': 'offensive'}, {'span': 'lose with it', 'label': 'offensive'}]
[{'span': 'paki', 'label': 'hatespeech'}, {'span': 'raped by paki muslims', 'label': 'hatespeech'}, {'span': 'aedophilia', 'label': 'hatespeech'}]
[{'span': 'race traitor', 'label': 'hatespeech'}, {'span': 'h0m0', 'label': 'offensive'}, {'span': 'igger', 'label': 'offensive'}]
[{'span': 'uzzie', 'label': 'hatespeech'}, {'span': 'take your house', 'label': 'offensive'}]
[{'span': 'isted', 'label': 'offensive'}, {'span': 'nigger', 'label': 'hatespeech'}]
[{'span': 'iglets', 'label

In [24]:
data=json.load(open(f"../output/HateXplain/FT/llama3.3-70.json"))
for item in data:
    item["pred"]=[x["span"] for x in item["pred"]]
   

with open("../output/HateXplain/FT/llama3.3-70.json","w") as f:
    json.dump(data,f,indent=2)    